In [ ]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(123)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


In [ ]:
# Initialize VSFCOntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsfc.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = VSFCOntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.9),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.4),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


In [4]:
v, _, matches = ontengine.getvector("thất vọng", debug=True)
print("Vector Neg:", v[20]) # Index 20 là NegativePolarity
if matches: print("Matched Data:", matches[0]['data']['polarities'])

Vector Neg: 0.99604917
Matched Data: ['NegativePolarity']


In [5]:
def test_vector_24d(text, engine):
    print(f"--- ANALYZING: '{text}' ---")
    
    # ---------------------------------------------------------
    # SỬA LỖI: Dùng getvector với debug=True thay vì get_matches
    # Output trả về: (Vector kết quả, Danh sách Token, Trace các từ match được)
    # ---------------------------------------------------------
    vector, tokens, matches = engine.getvector(text, debug=True)
    
    print(f"\n1. TOKENS: {tokens}")
    
    print(f"\n2. MATCHED PHRASES (Lexicon Hits):")
    if not matches:
        print("   (No phrases found in Ontology)")
    else:
        for m in matches:
            # m cấu trúc: {'phrase': 'từ tìm thấy', 'data': {chi tiết điểm số...}}
            print(f"   ✅ Phrase: '{m['phrase']}'")
            print(f"      -> Is Negation? {m['data']['isnegation']}")
            print(f"      -> Score: {m['data']['score']}")
            print(f"      -> Emotions: {m['data']['emotions']}")

    print(f"\n3. FINAL VECTOR SHAPE: {vector.shape}")
    print(f"   (Vector này sẽ được nối vào PhoBERT)")
    
    print(f"\n4. EMOTION VALUES (7 Dimensions):")
    # Mapping lại tên để dễ nhìn
    labels = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
    for i, label in enumerate(labels):
        print(f"   - {label}: {vector[i]:.4f}")

# --- CHẠY THỬ LẠI ---
test_text = "Món ăn này không ngon lắm nhưng phục vụ rất tuyệt vời"
if 'ontengine' in globals():
    test_vector_24d(test_text, ontengine)
else:
    print("⚠️ Hãy chạy Cell khởi tạo OntologyEngine trước!")

--- ANALYZING: 'Món ăn này không ngon lắm nhưng phục vụ rất tuyệt vời' ---

1. TOKENS: ['món', 'ăn', 'này', 'không', 'ngon', 'lắm', 'nhưng', 'phục vụ', 'rất', 'tuyệt vời']

2. MATCHED PHRASES (Lexicon Hits):
   ✅ Phrase: 'không'
      -> Is Negation? True
      -> Score: 0.95
      -> Emotions: {'NegationCue': 1.0}
   ✅ Phrase: 'rất'
      -> Is Negation? False
      -> Score: 0.372
      -> Emotions: {'Happiness': 1.0}
   ✅ Phrase: 'tuyệt vời'
      -> Is Negation? False
      -> Score: 0.5775
      -> Emotions: {'Happiness': 1.0}

3. FINAL VECTOR SHAPE: (24,)
   (Vector này sẽ được nối vào PhoBERT)

4. EMOTION VALUES (7 Dimensions):
   - Anger: 0.0000
   - Disgust: 0.0000
   - Fear: 0.0000
   - Happiness: 0.4266
   - Neutral: 0.0000
   - Sadness: 0.0000
   - Surprise: 0.0000


In [6]:
# ---------------------------------------------------------
# CELL 3: DATASET & PREPROCESSING (UIT-VSFC)
# ---------------------------------------------------------
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.utils.class_weight import compute_class_weight
import os

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
BASE_PATH = "../../data/vsfc/UIT-VSFC"

def load_vsfc_folder(folder_name):
    """Hàm đọc dữ liệu chuẩn VSFC"""
    sent_path = f"{BASE_PATH}/{folder_name}/sents.txt"
    label_path = f"{BASE_PATH}/{folder_name}/sentiments.txt"
    try:
        with open(sent_path, 'r', encoding='utf-8') as f:
            sentences = [line.strip() for line in f.readlines()]
        with open(label_path, 'r', encoding='utf-8') as f:
            # 0: Negative, 1: Neutral, 2: Positive
            labels = [int(line.strip()) for line in f.readlines()]
        
        min_len = min(len(sentences), len(labels))
        return pd.DataFrame({'Sentence': sentences[:min_len], 'Sentiment': labels[:min_len]})
    except FileNotFoundError:
        print(f"❌ Không tìm thấy thư mục {folder_name} tại {BASE_PATH}")
        # Trả về DataFrame rỗng để không crash code
        return pd.DataFrame({'Sentence': [], 'Sentiment': []})

# 2. Load Data
df_train = load_vsfc_folder("train")
df_valid = load_vsfc_folder("dev")
df_test = load_vsfc_folder("test")

print(f"✅ Loaded: Train={len(df_train)}, Dev={len(df_valid)}, Test={len(df_test)}")

# Kiểm tra xem data có load được không
if len(df_train) == 0:
    print("⚠️ CẢNH BÁO: Không load được dữ liệu. Hãy kiểm tra lại đường dẫn BASE_PATH!")
else:
    class VSFC_OntologyDataset(Dataset):
        def __init__(self, df, tokenizer, engine, max_len=128):
            self.df = df
            self.tokenizer = tokenizer
            self.engine = engine
            self.max_len = max_len
            
        def __len__(self): return len(self.df)
        
        def __getitem__(self, idx):
            txt = str(self.df.iloc[idx]['Sentence'])
            # Đảm bảo nhãn là int
            lbl = int(self.df.iloc[idx]['Sentiment']) 
            
            enc = self.tokenizer.encode_plus(
                txt, max_length=self.max_len, padding='max_length', 
                truncation=True, return_tensors='pt'
            )
            
            # Gọi Engine để lấy vector KG
            vec = self.engine.getvector(txt)
            
            return {
                'input_ids': enc['input_ids'].flatten(),
                'attention_mask': enc['attention_mask'].flatten(),
                'ontology_features': torch.tensor(vec, dtype=torch.float),
                'labels': torch.tensor(lbl, dtype=torch.long)
            }

    # DataLoaders
    # Lưu ý: ontengine phải được khởi tạo từ Cell 2 trước đó
    if 'ontengine' in globals():
        train_loader = DataLoader(VSFC_OntologyDataset(df_train, tokenizer, ontengine), batch_size=32, shuffle=True)
        valid_loader = DataLoader(VSFC_OntologyDataset(df_valid, tokenizer, ontengine), batch_size=32)
        test_loader = DataLoader(VSFC_OntologyDataset(df_test, tokenizer, ontengine), batch_size=32)

        # Class Weights
        train_labels = df_train['Sentiment'].tolist()
        cls_w = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
        
        # Đưa về GPU nếu có
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
        print(f"⚖️ Class Weights (Neg, Neu, Pos): {weights_tensor}")
    else:
        print("❌ LỖI: Biến 'ontengine' chưa tồn tại. Vui lòng chạy lại Cell 2!")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Loaded: Train=11426, Dev=1583, Test=3166
⚖️ Class Weights (Neg, Neu, Pos): tensor([0.7152, 8.3159, 0.6749], device='cuda:0')


In [7]:
# =========================================================
# CELL: INSPECT VECTOR DISTRIBUTION (DATAFRAME VIEW)
# Mục tiêu: Liệt kê vector của nhiều câu dưới dạng bảng để dễ so sánh.
# =========================================================
import pandas as pd
import numpy as np

# 1. Định nghĩa tên cột cho 24 chiều
COLS_24D = (
    ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"] + # 0-6
    ["Pleas", "Unpleas", "Loss", "Danger", "Unpred", "Sudden", "GoalObs", "Unfair", "Other"] + # 7-15
    ["Int_Hi", "Int_Me", "Int_Lo"] + # 16-18
    ["Pos", "Neg", "Neu"] + # 19-21
    ["NegHit", "NegCnt"] # 22-23
)

def inspect_vectors(texts, engine):
    data = []
    for txt in texts:
        # Lấy vector
        vec = engine.getvector(txt)
        # Làm tròn để dễ nhìn
        vec_rounded = np.round(vec, 2)
        
        # Tạo row
        row = {"Sentence": txt}
        for i, val in enumerate(vec_rounded):
            # Chỉ lưu các giá trị khác 0 để bảng gọn (hoặc lưu hết tuỳ ý)
            # Ở đây lưu hết để thấy cấu trúc
            row[COLS_24D[i]] = val
        data.append(row)
    
    # Tạo DataFrame
    df_results = pd.DataFrame(data)
    
    # Style: Tô màu các ô khác 0 để nổi bật
    def highlight_nonzero(val):
        color = '#d1e7dd' if isinstance(val, (int, float)) and abs(val) > 0.01 else '' # Màu xanh nhạt
        if isinstance(val, (int, float)) and val > 0.8: color = '#198754; color: white' # Xanh đậm nếu score cao
        if isinstance(val, (int, float)) and val < -0.5: color = '#dc3545; color: white' # Đỏ nếu âm (hiếm gặp vì tanh)
        return f'background-color: {color}'

    return df_results.style.map(highlight_nonzero, subset=COLS_24D)

# --- A. TEST TRÊN CÁC TRƯỜNG HỢP CỤ THỂ ---
samples = [
    "Món này ngon tuyệt vời",                  # Positive + Patch
    "Thái độ phục vụ quá tệ hại",             # Negative + Patch + Intensifier
    "Không ngon lắm",                         # Negation + Intensifier
    "Món ăn bình thường, không có gì đặc sắc", # Mixed / Neutral
    "Tôi rất thất vọng về cách làm việc",     # Sadness + Patch
    "Sợ quá đi mất",                          # Fear + Intensifier
    "Tôi chẳng thấy buồn cười gì cả"          # Negation strong
]

print(">>> A. MANUAL SAMPLES INSPECTION:")
display(inspect_vectors(samples, ontengine))

# --- B. TEST TRÊN DATASET THỰC TẾ (Nếu đã load df_train) ---
if 'df_train' in globals() and len(df_train) > 0:
    print("\n>>> B. RANDOM SAMPLES FROM TRAINING DATA:")
    random_samples = df_train['Sentence'].sample(5).tolist()
    display(inspect_vectors(random_samples, ontengine))
else:
    print("\n(Chưa load df_train nên bỏ qua phần B)")

>>> A. MANUAL SAMPLES INSPECTION:


,Sentence,Anger,Disgust,Fear,Happiness,Neutral,Sadness,Surprise,Pleas,Unpleas,Loss,Danger,Unpred,Sudden,GoalObs,Unfair,Other,Int_Hi,Int_Me,Int_Lo,Pos,Neg,Neu,NegHit,NegCnt
0,Món này ngon tuyệt vời,0.000000,0.000000,0.000000,0.600000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.940000,0.000000,0.000000,0.000000,0.000000
1,Thái độ phục vụ quá tệ hại,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Không ngon lắm,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
3,"Món ăn bình thường, không có gì đặc sắc",0.000000,0.000000,0.000000,0.000000,0.640000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.740000,1.000000,0.000000
4,Tôi rất thất vọng về cách làm việc,0.000000,0.000000,0.000000,0.420000,0.000000,0.850000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.810000,1.000000,0.000000,0.000000,0.000000
5,Sợ quá đi mất,0.000000,0.000000,0.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.980000,0.000000,0.000000,0.000000
6,Tôi chẳng thấy buồn cười gì cả,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



>>> B. RANDOM SAMPLES FROM TRAINING DATA:


,Sentence,Anger,Disgust,Fear,Happiness,Neutral,Sadness,Surprise,Pleas,Unpleas,Loss,Danger,Unpred,Sudden,GoalObs,Unfair,Other,Int_Hi,Int_Me,Int_Lo,Pos,Neg,Neu,NegHit,NegCnt
0,cô giảng hướng dẫn nhiệt tình .,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,giảng viên có tầm hiểu biết sâu rộng .,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,"ít hướng dẫn sinh viên thực hành , ra bài rồi để đó , toàn để sinh viên tự làm , đi học cũng như không đi , ai làm gì đó làm .",0.000000,0.000000,0.000000,0.000000,0.640000,0.480000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.870000,0.740000,0.000000,0.000000
3,nội dung hơi khó hiểu .,0.000000,0.000000,0.000000,0.000000,0.000000,0.470000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.860000,0.000000,0.000000,0.000000
4,nhiệt tình và rất chú tâm trong công việc giảng dạy đối với sinh viên .,0.000000,0.000000,0.000000,0.810000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.990000,0.000000,0.000000,0.000000,0.000000


In [8]:
# Import PhoBERT Fusion architecture from src.models
from src.models.phobert_fusion_models import PhoBERT_Fusion_V2

try:
    dummy_vec = ontengine.getvector('test')
    if isinstance(dummy_vec, tuple): dummy_vec = dummy_vec[0]
    ONT_INPUT_DIM = len(dummy_vec)
except:
    ONT_INPUT_DIM = 24
print(f'✅ Loaded PhoBERT_Fusion_V2 | Detected Ontology Dimension: {ONT_INPUT_DIM}')


📏 Detected Ontology Dimension: 24
✅ Model Architecture defined successfully!


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
a2a-sdk 0.3.22 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.1 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.

In [10]:
# ---------------------------------------------------------
# CELL 5: TRAINER (FIXED IMPORTS)
# ---------------------------------------------------------
import copy
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

# Đảm bảo device đã được define
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_experiment(model, name, epochs=15, patience=4):
    print(f"\n🚀 START: {name}")
    
    # Bọc DataParallel nếu có nhiều GPU
    if torch.cuda.device_count() > 1: 
        model = nn.DataParallel(model)
    model.to(device)
    
    optim = torch.optim.AdamW(model.parameters(), lr=2e-5)
    
    # Đảm bảo weights_tensor đã có (từ Cell 3). Nếu chưa thì dùng None (không cân bằng)
    if 'weights_tensor' in globals():
        crit = nn.CrossEntropyLoss(weight=weights_tensor)
    else:
        print("⚠️ Warning: 'weights_tensor' not found. Training without class weighting.")
        crit = nn.CrossEntropyLoss()
    
    best_f1, pat = 0, 0
    
    # Deepcopy weights ban đầu
    if isinstance(model, nn.DataParallel):
        best_w = copy.deepcopy(model.module.state_dict())
    else:
        best_w = copy.deepcopy(model.state_dict())
    
    for ep in range(epochs):
        model.train()
        loss_acc = 0
        
        # Train Loop
        for b in train_loader:
            optim.zero_grad()
            # Move batch to device
            ids = b['input_ids'].to(device)
            m = b['attention_mask'].to(device)
            o = b['ontology_features'].to(device)
            l = b['labels'].to(device)
            
            # Forward & Backward
            out = model(ids, m, o)
            loss = crit(out, l)
            loss.backward()
            optim.step()
            loss_acc += loss.item()
            
        # Validation
        acc, f1 = evaluate(model, valid_loader)
        avg_loss = loss_acc / len(train_loader)
        print(f"Ep {ep+1} | Loss: {avg_loss:.4f} | Val F1: {f1:.4f}", end=" ")
        
        # Early Stopping Check
        if f1 > best_f1:
            best_f1 = f1
            pat = 0
            if isinstance(model, nn.DataParallel):
                best_w = copy.deepcopy(model.module.state_dict())
            else:
                best_w = copy.deepcopy(model.state_dict())
            print("✅")
        else:
            pat += 1
            print(f"⚠️ {pat}")
        
        if pat >= patience:
            print(f"🛑 Early Stopping triggered after {ep+1} epochs.")
            break
    
    # Load Best Weights
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_w)
    else:
        model.load_state_dict(best_w)
        
    return model

def evaluate(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids = b['input_ids'].to(device)
            m = b['attention_mask'].to(device)
            o = b['ontology_features'].to(device)
            l = b['labels'].to(device)
            
            out = model(ids, m, o)
            _, pr = torch.max(out, 1)
            p.extend(pr.cpu().numpy())
            t.extend(l.cpu().numpy())
            
    return accuracy_score(t, p), f1_score(t, p, average='macro')

In [11]:
# ---------------------------------------------------------
# CELL 6: MAIN EXPERIMENTS (5 CASES - VSFC)
# ---------------------------------------------------------
NC = 3 # Negative, Neutral, Positive

# 1. Baseline
m1 = PhoBERT_Fusion_V2(NC, 'none')
m1 = train_experiment(m1, "Baseline (PhoBERT)", epochs=15)

# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Raw + Gate", epochs=15)



2026-02-24 08:55:24.697810: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771923324.881186      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771923324.933846      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771923325.362462      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771923325.362503      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771923325.362506      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 START: Baseline (PhoBERT)


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Ep 1 | Loss: 0.5434 | Val F1: 0.7482 ✅
Ep 2 | Loss: 0.2992 | Val F1: 0.7981 ✅
Ep 3 | Loss: 0.2147 | Val F1: 0.8263 ✅
Ep 4 | Loss: 0.1746 | Val F1: 0.8461 ✅
Ep 5 | Loss: 0.1277 | Val F1: 0.8349 ⚠️ 1
Ep 6 | Loss: 0.1135 | Val F1: 0.8378 ⚠️ 2
Ep 7 | Loss: 0.1048 | Val F1: 0.8443 ⚠️ 3
Ep 8 | Loss: 0.0871 | Val F1: 0.8505 ✅
Ep 9 | Loss: 0.0913 | Val F1: 0.8268 ⚠️ 1
Ep 10 | Loss: 0.0801 | Val F1: 0.8338 ⚠️ 2
Ep 11 | Loss: 0.0802 | Val F1: 0.8216 ⚠️ 3
Ep 12 | Loss: 0.0701 | Val F1: 0.8245 ⚠️ 4
🛑 Early Stopping triggered after 12 epochs.


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 START: Raw + Gate
Ep 1 | Loss: 0.5259 | Val F1: 0.8118 ✅
Ep 2 | Loss: 0.3208 | Val F1: 0.8070 ⚠️ 1
Ep 3 | Loss: 0.2449 | Val F1: 0.8169 ✅
Ep 4 | Loss: 0.1889 | Val F1: 0.8344 ✅
Ep 5 | Loss: 0.1378 | Val F1: 0.8382 ✅
Ep 6 | Loss: 0.0986 | Val F1: 0.8348 ⚠️ 1
Ep 7 | Loss: 0.1342 | Val F1: 0.8121 ⚠️ 2
Ep 8 | Loss: 0.0977 | Val F1: 0.8440 ✅
Ep 9 | Loss: 0.0819 | Val F1: 0.8098 ⚠️ 1
Ep 10 | Loss: 0.0617 | Val F1: 0.8273 ⚠️ 2
Ep 11 | Loss: 0.0627 | Val F1: 0.8295 ⚠️ 3
Ep 12 | Loss: 0.0749 | Val F1: 0.8452 ✅
Ep 13 | Loss: 0.0606 | Val F1: 0.8426 ⚠️ 1
Ep 14 | Loss: 0.0568 | Val F1: 0.8546 ✅
Ep 15 | Loss: 0.0397 | Val F1: 0.8428 ⚠️ 1


In [12]:
import torch
import os

# Tạo thư mục lưu trữ nếu cần
output_dir = "/kaggle/working/checkpoints"
os.makedirs(output_dir, exist_ok=True)

# 1. Lưu checkpoint cho m1 (Baseline)
m1_path = os.path.join(output_dir, "phobert_baseline_m1.pth")
torch.save(m1.state_dict(), m1_path)
print(f"✅ Đã lưu m1 tại: {m1_path}")

# 2. Lưu checkpoint cho m3 (Raw + Gate)
m3_path = os.path.join(output_dir, "phobert_gate_m3.pth")
torch.save(m3.state_dict(), m3_path)
print(f"✅ Đã lưu m3 tại: {m3_path}")

# Hiển thị link để tải nhanh (dùng cho trình duyệt)
from IPython.display import FileLink
display(FileLink(r'checkpoints/phobert_baseline_m1.pth'))
display(FileLink(r'checkpoints/phobert_gate_m3.pth'))

✅ Đã lưu m1 tại: /kaggle/working/checkpoints/phobert_baseline_m1.pth
✅ Đã lưu m3 tại: /kaggle/working/checkpoints/phobert_gate_m3.pth


/kaggle/working/checkpoints/phobert_baseline_m1.pth

/kaggle/working/checkpoints/phobert_gate_m3.pth

In [ ]:

# 2. Raw + Concat (xAI)
m2 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=None)
m2 = train_experiment(m2, "Raw + Concat", epochs=15)

# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Raw + Gate", epochs=15)



Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 START: Raw + Concat
Ep 1 | Loss: 0.5234 | Val F1: 0.7847 ✅
Ep 2 | Loss: 0.3084 | Val F1: 0.8396 ✅
Ep 3 | Loss: 0.2414 | Val F1: 0.8341 ⚠️ 1
Ep 4 | Loss: 0.1735 | Val F1: 0.8504 ✅
Ep 5 | Loss: 0.1570 | Val F1: 0.8407 ⚠️ 1
Ep 6 | Loss: 0.1098 | Val F1: 0.8392 ⚠️ 2
Ep 7 | Loss: 0.1106 | Val F1: 0.8196 ⚠️ 3


In [ ]:
# 4. Dense (64d) + Concat
m4 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=64)
m4 = train_experiment(m4, "Dense(64) + Concat", epochs=15)

# 5. Dense (64d) + Gate
m5 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=64)
m5 = train_experiment(m5, "Dense(64) + Gate", epochs=15)

In [ ]:
# ---------------------------------------------------------
# CELL 7: FINAL REPORT (FIXED)
# ---------------------------------------------------------
import pandas as pd
from sklearn.metrics import classification_report, f1_score, accuracy_score # <--- THÊM DÒNG NÀY

def gp(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            _, pr = torch.max(model(ids, m, o), 1)
            p.extend(pr.cpu().numpy()); t.extend(l.cpu().numpy())
    return t, p

# Dự đoán cho tất cả các model đã train
y_true, p1 = gp(m1, test_loader)
_, p2 = gp(m2, test_loader)
_, p3 = gp(m3, test_loader)
_, p4 = gp(m4, test_loader)
_, p5 = gp(m5, test_loader)

res = {
    "Model": ["Baseline", "Raw+Concat", "Raw+Gate", "Dense+Concat", "Dense+Gate"],
    "F1-Macro": [f1_score(y_true, p, average='macro') for p in [p1, p2, p3, p4, p5]],
    "Accuracy": [accuracy_score(y_true, p) for p in [p1, p2, p3, p4, p5]]
}

print("\n" + "="*50)
print("🏆 UIT-VSFC FINAL RESULTS")
print("="*50)
print(pd.DataFrame(res).round(4).to_markdown(index=False))

target_names = ["Negative", "Neutral", "Positive"] # 0, 1, 2
print("\n" + "="*50)
print("📑 DETAILED REPORT (DENSE + GATE)")
print("="*50)
# Bây giờ hàm này đã được định nghĩa và sẽ chạy bình thường
print(classification_report(y_true, p5, target_names=target_names, digits=4))

In [ ]:
# =========================================================
# CELL: EVALUATION - BASELINE + RULE-BASE (NEURO-SYMBOLIC)
# Kết hợp mô hình Baseline (PhoBERT) với Luật Ontology
# =========================================================
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

# 1. Định nghĩa lớp Reasoner (Neuro-Symbolic) để kết hợp Neural + Rule
class NeuroSymbolicReasoner:
    def __init__(self, model, engine, tokenizer, device, 
                 ontology_threshold=0.6, model_conf_limit=0.85):
        self.model = model
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.ont_threshold = ontology_threshold
        self.model_conf_limit = model_conf_limit
        self.idx2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

    def _get_ontology_signals(self, vector):
        # Lấy tín hiệu từ vector tri thức (Cấu trúc v12.2): Pos (Index 19), Neg (Index 20)
        pos_score = vector[19]
        neg_signal = vector[20]
        emotions = vector[0:7]
        max_emo_idx = np.argmax(emotions)
        return pos_score, neg_signal, max_emo_idx

    def infer(self, text):
        # B1: Neural Inference - Sử dụng mô hình Baseline m1
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding='max_length')
        input_ids = inputs['input_ids'].to(self.device)
        attention_mask = inputs['attention_mask'].to(self.device)
        
        # Lấy vector tri thức từ OntologyEngine
        vec, _, _ = self.engine.getvector(text, debug=True)
        # m1 có fusion_type='none' nên sẽ bỏ qua tham số ontology_features này trong forward
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)
            
        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        neural_label = self.idx2label[pred_idx]

        # B2: Symbolic Override (Hiệu chỉnh bằng luật cứng)
        pos_signal, neg_signal, _ = self._get_ontology_signals(vec)
        final_label = neural_label
        
        # LUẬT: Nếu Baseline dự đoán là Neutral nhưng Ontology phát hiện từ ngữ mang tính cực đoan
        # và độ tin cậy của mô hình Neural không quá cao, ta sẽ ưu tiên tri thức Ontology.
        if pred_idx == 1: # Model dự đoán Neutral
            if neg_signal > self.ont_threshold and confidence < self.model_conf_limit:
                final_label = "Negative"
            elif pos_signal > self.ont_threshold and confidence < self.model_conf_limit:
                final_label = "Positive"

        return neural_label, final_label

# 2. Kiểm tra biến m1 và Chạy đánh giá
if 'm1' not in globals():
    print("❌ LỖI: Biến 'm1' chưa được định nghĩa. Hãy chắc chắn bạn đã chạy Cell 11 để huấn luyện Baseline!")
else:
    print(f"🚀 Bắt đầu đánh giá Baseline + Rule-base trên {len(df_test)} mẫu test...")

    reasoner = NeuroSymbolicReasoner(
        model=m1, 
        engine=ontengine, 
        tokenizer=tokenizer, 
        device=device,
        ontology_threshold=0.55 # Ngưỡng để kích hoạt luật hiệu chỉnh
    )

    y_true = []
    y_pred_baseline = [] # Kết quả nguyên bản của PhoBERT
    y_pred_neuro = []    # Kết quả sau khi áp dụng luật tri thức

    for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
        text = row['Sentence']
        true_label = int(row['Sentiment'])
        
        # Lấy dự đoán từ cả hai chế độ
        base_lbl, neuro_lbl = reasoner.infer(text)
        
        label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
        y_true.append(true_label)
        y_pred_baseline.append(label_map[base_lbl])
        y_pred_neuro.append(label_map[neuro_lbl])

    # 3. Tổng hợp và Hiển thị kết quả
    acc_base = accuracy_score(y_true, y_pred_baseline)
    f1_base = f1_score(y_true, y_pred_baseline, average='macro')
    acc_neuro = accuracy_score(y_true, y_pred_neuro)
    f1_neuro = f1_score(y_true, y_pred_neuro, average='macro')

    results_df = pd.DataFrame({
        "Metric": ["Accuracy", "F1-Macro"],
        "Baseline (PhoBERT Only)": [acc_base, f1_base],
        "Neuro-Symbolic (Baseline + Rules)": [acc_neuro, f1_neuro],
        "Improvement": [acc_neuro - acc_base, f1_neuro - f1_base]
    })

    print("\n" + "="*60)
    print("🏆 KẾT QUẢ SO SÁNH (BASELINE vs. NEURO-SYMBOLIC)")
    print("="*60)
    print(results_df.round(4).to_markdown(index=False))

    # Đếm số lượng mẫu đã được luật can thiệp sửa đổi
    changes = sum(1 for b, n in zip(y_pred_baseline, y_pred_neuro) if b != n)
    print(f"\n🔄 Tổng số mẫu được luật can thiệp sửa lại: {changes}/{len(df_test)} ({changes/len(df_test)*100:.2f}%)")